In [1]:
pip install tensorflow

In [2]:
pip install keras

In [3]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [4]:
#importing libraries
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score,classification_report
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import nltk 
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string

In [5]:
#loading data
df = pd.read_csv("data_set.csv")
df.shape

(700, 2)

In [6]:
df.isnull().sum()

message     0
category    0
dtype: int64

In [7]:
df.duplicated().sum()
#df = df.drop_duplicates(keep='first')


0

In [8]:
df.shape
df.head()

,message,category
0,Session expirée trop rapidement. Redémarrage d...,Authentication
1,Authentication failed repeatedly. Please assist.,Authentication
2,Authentication failed repeatedly. Any solution...,Authentication
3,Forgot my password.,Authentication
4,Access request pending. This needs urgent atte...,Authentication


In [9]:
#preprocessing the data
#convertir les deux colones en minuscule
df["message"] = df["message"].str.lower()
df["category"] = df["category"].str.lower()

In [10]:
#checking balance 
df.groupby('category').describe()

message         \
                 count unique   
category                        
authentication     100    100   
email              100    100   
hardware           100    100   
network            100    100   
printing           100    100   
security           100    100   
software           100    100   

                                                                        
                                                              top freq  
category                                                                
authentication  session expirée trop rapidement. redémarrage d...    1  
email                                   messages stuck in outbox.    1  
hardware        le port usb ne fonctionne plus. besoin d'assis...    1  
network         le wifi se déconnecte souvent. déjà signalé, t...    1  
printing                      paper jam. already tried rebooting.    1  
security        security patch failed. reported before, still ...    1  
software                       l'application plante au lancement.    1

In [11]:
#remove ponctuation
df["message"] = df["message"].str.replace(r'[^\w\s]', '', regex=True)

#remove numeric characters 
df['message'] = df['message'].str.replace(r'\d+', '', regex=True)

nltk.download('punkt_tab')#punkt est un modele qui permet nltk.word_tokenize de decouper le texte de facon intelligente
nltk.download('stopwords')#stopwords est une liste qui contient les mots tres frequents et inutiles comme {the,my...}

ps = PorterStemmer()#un objet qui permet de transformer les mots a leurs racines running ----> run

def transform_text(text):
    text = text.lower() #transforme tout le texte en mniniscule
    text = nltk.word_tokenize(text) # permet un  découpage avancé  du texte en prenant en consideration la ponctuation et plus chose
    
    y = []
    for i in text:
        if i.isalnum(): #verifie is le mot est alphanumerique s'il est un caractere special il va l'ignoré par exemple:$ ou & ...
            y.append(i)
    
    text = y[:] #affecte a text le contenu entier de la liste y
    y.clear()   #efface le contenu de y

    #verification si le mot n'est pas inutil et n'est pas un caractere de ponctuation pour l'ajouter a la liste y qui est vide
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i) #stopwords.words('english') :liste des mots inutiles en anglais mais qui son tres frequents {or,the ,this ....}
                        #chaine.punctuation: une chaine qui contient tous les caractes speciaux 
            
    text = y[:] #donner a text le contenu de y a nouveau
    y.clear()   #vider y
            
    for i in text:
        y.append(ps.stem(i)) # transformer chaque mot a son origine puis le stocker dans y 
            
                    
    return " ".join(y) #refusionner le text en utilisant l'espace comme separateur

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
df["message"] = df["message"].apply(transform_text)

In [13]:
df["category"].value_counts()

category
authentication    100
email             100
network           100
security          100
hardware          100
printing          100
software          100
Name: count, dtype: int64

In [14]:
#Encoding
"""category_map = {
    "Authentication": 0,
    "Email": 1,
    "Hardware": 2,
    "Network": 3,
    "Printing": 4,
    "Security": 5,
    "Software": 6 
}
df["labels"] = df["category"].map(category_map)"""

label_Encoder = LabelEncoder()
df["label"] = label_Encoder.fit_transform(df["category"])

#saving the Encoder using joblib
import joblib as jb

jb.dump(label_Encoder,'label_Encoder.pkl')

     

['label_Encoder.pkl']

In [15]:
#Tokenisation
tokenizer = Tokenizer(num_words=5000, oov_token = "<OOV>")

tokenizer.fit_on_texts(df["message"])
import pickle

with open("tokenizer.pickle", "wb") as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)


#tokenizer.word_index #the dictionnary that contains each word with a unique number

sequences = tokenizer.texts_to_sequences(df['message'])

padded = pad_sequences(sequences, maxlen=70)


In [16]:
X_train,X_test,y_train,y_test = train_test_split(padded,df["label"],test_size = 0.2,random_state = 42)


In [17]:
# Build model
model = keras.Sequential([
    keras.layers.Embedding(input_dim=5000, output_dim=128, input_length=70),
    keras.layers.Conv1D(64, kernel_size=3, activation="relu"),
    keras.layers.MaxPooling1D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(7, activation="softmax")
])
# Compile model
model.compile(loss='sparse_categorical_crossentropy', optimizer="adam", metrics=["accuracy"])

C:\Users\User\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
# Train model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))
print(history.history['accuracy'])       # Training accuracy per epoch
print(history.history['val_accuracy'])   # Validation (test) accuracy per epoch


Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.1273 - loss: 1.9496 - val_accuracy: 0.1929 - val_loss: 1.9414
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.2949 - loss: 1.9241 - val_accuracy: 0.2643 - val_loss: 1.9160
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.4060 - loss: 1.8744 - val_accuracy: 0.4071 - val_loss: 1.8292
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6052 - loss: 1.7043 - val_accuracy: 0.6286 - val_loss: 1.5026
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.7364 - loss: 1.2493 - val_accuracy: 0.8071 - val_loss: 0.9096
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.8704 - loss: 0.7129 - val_accuracy: 0.9286 - val_loss: 0.4457
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9618 - loss: 0.3145 - val_accuracy: 0.9500 - val_loss: 0.2423
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9814 - loss: 0.1998 - val_accuracy: 0.9643 - v

In [19]:
model.save("tf_nl_classifier.h5")

In [20]:
#msg = ["j'ai oublie mon mot de passe"]
#msg = ["je recois pas les emails"]
#msg = ["l'ancre sur papiers n'est pas claires"]
msg = ["je recois pas des email"]
msg_nettoye = [transform_text(item) for item in msg]
seq  = tokenizer.texts_to_sequences(msg_nettoye)
pad = pad_sequences(seq, maxlen=70)
pred = model.predict(pad)
print(pred)
label = label_Encoder.inverse_transform([np.argmax(pred)])
print("Predicted category:", label[0])



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step
[[0.45650777 0.41305965 0.00367626 0.00788177 0.07936969 0.02393106
  0.01557375]]
Predicted category: authentication


In [21]:
model.save("chatbot_model.h5")
